# Testing LLM APIs
Welcome to this practical session. In this notebook, we will explore how to interact with the production OpenAI API, manipulate hyperparameters like temperature, and build automated testing assertions for validating structured responses.

In [1]:
import os
import os.path as osp

from pathlib import Path
from pprint import pprint

import sys

root = Path.cwd().parent 
if str(root) not in sys.path:    
    sys.path.append(str(root))

In [2]:
from src.config import CFG

## Installation & Environment Setup
First, we need to install the official OpenAI SDK and configure our secure API token environment variable.

In [4]:
import time
import os
import json
from openai import OpenAI
import os 

# Instruct students to input their real OpenAI API key

# Initialize the standard production client
client = OpenAI(api_key=CFG.OPENAI_API_KEY)

print("OpenAI Production Client Initialized!")

OpenAI Production Client Initialized!


In [14]:
from pprint import pprint

## Managing Hyperparameters: Temperature Determinism
Let's see how setting a low temperature (highly deterministic) vs. a high temperature (highly creative) impacts raw API output variability.

In [5]:
prompt = "Qu'est-ce qu'un agent IA? réponds en quelques lignes"

outputs = []
print("--- Testing Deterministic Output (temperature=0.0) ---")
for i in range(2):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.
    )
    print(f"Run {i+1}:\n")
    response = response.choices[0].message.content.strip()
    pprint(response)
    outputs.append(response)
    print()

--- Testing Deterministic Output (temperature=0.0) ---
Run 1:

('Un agent IA (intelligence artificielle) est un système informatique capable '
 "d'effectuer des tâches de manière autonome en utilisant des algorithmes "
 "d'apprentissage automatique, de traitement du langage naturel ou d'autres "
 "techniques d'IA. Ces agents peuvent analyser des données, prendre des "
 "décisions, interagir avec les utilisateurs et s'adapter à leur "
 'environnement. Ils sont utilisés dans divers domaines, tels que les '
 'assistants virtuels, les systèmes de recommandation et la robotique.')

Run 2:

('Un agent IA (intelligence artificielle) est un système informatique capable '
 "d'effectuer des tâches de manière autonome en utilisant des algorithmes "
 "d'apprentissage automatique, de traitement du langage naturel ou d'autres "
 "techniques d'IA. Ces agents peuvent analyser des données, prendre des "
 "décisions, interagir avec les utilisateurs et s'adapter à leur "
 'environnement. Ils sont utilisé

In [6]:
from src.utils import calc_similarity

In [7]:
calc_similarity(outputs[0], outputs[1])

TF-IDF Cosine Similarity: 1.0000


np.float64(0.9999999999999997)

In [8]:
print("\n--- Testing Creative Output (temperature=1.2) ---")
outputs = []

for i in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=1.2
    )
    print(f"Run {i+1}:\n")
    response = response.choices[0].message.content.strip()
    pprint(response)
    outputs.append(response)
    print()


--- Testing Creative Output (temperature=1.2) ---
Run 1:

('Un agent IA (intelligence artificielle) est un logiciel ou un système conçu '
 'pour effectuer des tâches de manière autonome ou semi-autonome en utilisant '
 "des techniques d'intelligence artificielle. Cela inclut des capacités comme "
 "la perception, la prise de décision, l'apprentissage et l'interaction avec "
 "les utilisateurs ou d'autres systèmes. Les agents IA peuvent être utilisés "
 "dans divers domaines, tels que l'assistance virtuelle, la robotique, la "
 "reconnaissance vocale, et l'analyse de données, pour améliorer l'efficacité "
 'et automatiser des processus.')

Run 2:

("Un agent IA (intelligence artificielle) est un système capable d'accomplir "
 'des tâches autonomes ou semi-autonomes en utilisant des algorithmes et des '
 'données pour traiter des informations, prendre des décisions et interagir '
 'avec son environnement. Ces agents peuvent varier en complexité, allant de '
 'simples bots automatisés à 

In [9]:
calc_similarity(outputs[0], outputs[1])

TF-IDF Cosine Similarity: 0.5364


np.float64(0.5363774522438594)

In [27]:
response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": prompt}],
        # temperature=0.3,
        reasoning_effort="low"
    )

response = response.choices[0].message.content.strip()
print(response)


Un agent IA est un système d’intelligence artificielle capable de **percevoir son environnement, prendre des décisions et agir de manière autonome** pour atteindre un objectif.  
Contrairement à un simple chatbot, il peut souvent **enchaîner plusieurs actions**, utiliser des outils, et s’adapter à la situation.  
Exemple : un agent IA peut planifier un voyage, envoyer des emails ou gérer certaines tâches à votre place.


In [30]:
prompt = "Qu'est-ce qu'un agent IA? réponds avec 300 mots"

response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "user", "content": prompt}],
    reasoning_effort="low",
    stream=True  # Enables streaming
)

for chunk in response:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)

print() 

Un agent IA, ou agent intelligent, est un système informatique capable de percevoir son environnement, de traiter des informations, puis d’agir de manière autonome pour atteindre un objectif précis. Contrairement à un programme classique qui exécute seulement des instructions fixes, un agent IA peut souvent s’adapter à la situation, prendre des décisions et parfois apprendre de ses expériences.

On peut le voir comme un “assistant” numérique qui observe, réfléchit et agit. Par exemple, un agent IA peut analyser des données, répondre à des messages, recommander un produit, conduire une voiture, ou encore gérer des tâches répétitives dans une entreprise. Dans certains cas, il interagit avec des outils externes comme un moteur de recherche, une base de données, un calendrier ou une application de messagerie.

Un agent IA suit généralement un cycle simple : il reçoit des entrées, les interprète, choisit une action, puis observe le résultat pour ajuster son comportement. Ce fonctionnement l